In [1]:
import pandas as pd
import numpy as np

from scipy import stats

import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.stats.weightstats import zconfint

In [2]:
df = pd.read_csv(
    "../data/raw/df_clean_one.csv",
    parse_dates=["Date"]
)

## Confidence Interval

In [3]:
# variable of interest 

close_price = df["Close"]

>### Assumptions
- Random observations
- Continuous variable
- Large sample size (n > 30)

In [4]:
mean_close = close_price.mean()

ci_low, ci_high = stats.t.interval(
    confidence=0.95,
    df=len(close_price)-1,
    loc=mean_close,
    scale=stats.sem(close_price)
)

print(f"Mean Close Price: {mean_close:.2f}")
print(f"95% CI: ({ci_low:.2f}, {ci_high:.2f})")

Mean Close Price: 76.34
95% CI: (75.84, 76.84)


>> Interpretation Template

We are 95% confident that the true mean closing stock price lies between the lower and upper confidence limits.

## One-Sample t-Test

>> Hypotheses
>>>Null
H0 :μ=100

>>>Alternative
H1:μ=100

In [16]:
t_stat, p_value = stats.ttest_1samp(
    df["Close"],
    popmean=75
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: 5.259288850355159
P-value: 1.4470915382943534e-07


In [22]:
alpha = 0.05

if p_value < alpha:
    print("Reject H0")
else:
    print("Fail to Reject H0")

Reject H0


## Independent Two-Sample t-Test



In [7]:
df["USA_Group"] = np.where(
    df["Country"] == "usa",
    "USA",
    "Non-USA"
)

>> Research Question

Is there a difference in average closing prices between USA and Non-USA companies?

>> Hypotheses
Null

H0:μUSA=μNon−USA
	​
Alternative
H1:μUSA=μNon−USA

#### Assumptions
- Normality

In [8]:
stats.shapiro(
    df.sample(5000)["Close"]
)

ShapiroResult(statistic=np.float64(0.47012877744997406), pvalue=np.float64(2.0783008953109377e-81))

- Equal Variance

In [9]:
usa = df[df["USA_Group"]=="USA"]["Close"]

non_usa = df[df["USA_Group"]=="Non-USA"]["Close"]

stats.levene(
    usa,
    non_usa
)

LeveneResult(statistic=np.float64(4624.120362798224), pvalue=np.float64(0.0))

>>> Test

In [10]:
t_stat, p_value = stats.ttest_ind(
    usa,
    non_usa,
    equal_var=False
)

print(t_stat)
print(p_value)

130.6069416637685
0.0


## One-Way ANOVA

In [11]:
top_industries = (
    df["Industry_Tag"]
    .value_counts()
    .head(5)
    .index
)

anova_df = df[
    df["Industry_Tag"].isin(top_industries)
]

####  Research Question

Do average closing stock prices differ across industries?

Hypotheses
Null
H0:μ1=μ2=μ3=⋯
Alternative

At least one industry mean differs.

#### Assumptions
Normality

In [12]:
for industry in top_industries:
    
    sample = anova_df[
        anova_df["Industry_Tag"] == industry
    ]["Close"]
    
    print(
        industry,
        stats.shapiro(
            sample.sample(
                min(500,len(sample))
            )
        )
    )

technology ShapiroResult(statistic=np.float64(0.5997993921751732), pvalue=np.float64(2.337412513377358e-32))
retail ShapiroResult(statistic=np.float64(0.6062488388121668), pvalue=np.float64(3.712880070561525e-32))
automotive ShapiroResult(statistic=np.float64(0.7548650018818435), pvalue=np.float64(1.3891171832933163e-26))
finance ShapiroResult(statistic=np.float64(0.7695179714305682), pvalue=np.float64(6.706129534960864e-26))
apparel ShapiroResult(statistic=np.float64(0.7910800292917648), pvalue=np.float64(7.879871337223828e-25))


- Homogeneity of Variance

In [13]:
groups = [
    
    anova_df[
        anova_df["Industry_Tag"] == industry
    ]["Close"]
    
    for industry in top_industries
]

stats.levene(*groups)

LeveneResult(statistic=np.float64(1049.3105743383546), pvalue=np.float64(0.0))

## ANOVA Test

In [14]:
f_stat, p_value = stats.f_oneway(
    *groups
)

print("F-statistic:", f_stat)
print("P-value:", p_value)

F-statistic: 1806.3423853163763
P-value: 0.0


Method 5: Linear Regression

This is the strongest parametric method for your report.

Research Question

Can opening stock prices significantly predict closing stock prices?

In [15]:
model = smf.ols(
    "Close ~ Open",
    data=df
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  Close   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 8.190e+08
Date:                Fri, 12 Jun 2026   Prob (F-statistic):               0.00
Time:                        21:22:03   Log-Likelihood:            -7.5448e+05
No. Observations:              310122   AIC:                         1.509e+06
Df Residuals:                  310120   BIC:                         1.509e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0297      0.006      5.284      0.0